# 02 — Contributor Groups and Code Ownership

Identify contributor groups from commit patterns, score code ownership per target,
and assess knowledge risk via bus factor.

**Outputs:** `data/processed/contributor_groups.parquet`, `data/processed/target_ownership.parquet`

In [ ]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import pandas as pd
import seaborn as sns
from community import community_louvain
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.metrics import adjusted_rand_score

from build_optimiser.config import Config
from build_optimiser.contributors import (
    build_contributor_target_matrix,
    cluster_contributors_hierarchical,
    cluster_contributors_nmf,
    compute_bus_factor,
    compute_ownership,
    normalise_to_distributions,
)
from build_optimiser.graph import attach_metrics, load_graph

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

cfg = Config.from_yaml('../config.yaml')
file_df   = pd.read_parquet(cfg.processed_data_dir / 'file_metrics.parquet')
target_df = pd.read_parquet(cfg.processed_data_dir / 'target_metrics.parquet')
edge_df   = pd.read_parquet(cfg.processed_data_dir / 'edge_list.parquet')
G = load_graph(cfg.processed_data_dir / 'edge_list.parquet')
attach_metrics(G, target_df)

## 1. Load Contributor Data

In [ ]:
commits_df = pd.read_parquet(cfg.processed_data_dir / 'contributor_target_commits.parquet')
contributors_df = pd.read_csv(cfg.raw_data_dir / 'contributors.csv')

print('contributor_target_commits schema:')
print(commits_df.dtypes)
print(f'Rows: {len(commits_df)}')
print(f'Contributors: {commits_df["contributor"].nunique()}')
print(f'Targets: {commits_df["cmake_target"].nunique()}')

print('\ncontributors.csv head:')
contributors_df.head()

## 2. Build Contributor-Target Matrix

In [ ]:
# Auto-tune thresholds for small datasets
contrib_totals = commits_df.groupby('contributor')['commit_count'].sum()
target_totals  = commits_df.groupby('cmake_target')['commit_count'].sum()

min_cc = min(10, int(contrib_totals.quantile(0.25))) if len(contrib_totals) > 0 else 0
min_tc = min(5,  int(target_totals.quantile(0.25)))  if len(target_totals) > 0 else 0

print(f'Thresholds: min_contributor_commits={min_cc}, min_target_commits={min_tc}')

matrix = build_contributor_target_matrix(
    commits_df,
    min_contributor_commits=min_cc,
    min_target_commits=min_tc,
)
print(f'Matrix shape: {matrix.shape}  (contributors x targets)')
if matrix.size > 0:
    print(f'Sparsity: {(matrix == 0).values.sum() / matrix.size * 100:.1f}%')

# Heatmap of top contributors vs top targets (clipped for readability)
if matrix.size > 0:
    top_contribs = matrix.sum(axis=1).nlargest(30).index
    top_targets  = matrix.sum(axis=0).nlargest(40).index
    sub = matrix.loc[top_contribs, top_targets]

    fig, ax = plt.subplots(figsize=(16, 8))
    sns.heatmap(
        np.log1p(sub), ax=ax, cmap='YlOrRd',
        xticklabels=True, yticklabels=True,
        linewidths=0.3, cbar_kws={'label': 'log(1+commits)'},
    )
    ax.set_title('Contributor-Target Commit Matrix (top 30 contributors x top 40 targets, log scale)')
    ax.set_xlabel('cmake_target')
    ax.set_ylabel('contributor')
    plt.xticks(rotation=45, ha='right', fontsize=7)
    plt.yticks(fontsize=8)
    plt.tight_layout()
    plt.show()
else:
    print('⚠ Empty matrix after filtering — skipping heatmap.')

## 3. Contributor Clustering

### 3a. Hierarchical Clustering (Ward + Jensen-Shannon)

In [ ]:
if matrix.shape[0] >= 2:
    hier_result = cluster_contributors_hierarchical(
        matrix,
        metric='jensenshannon',
        cut_levels=[3, 5, 7, 10],
    )

    Z     = hier_result['linkage_matrix']
    dendro = hier_result['dendrogram_data']
    hier_assignments = hier_result['assignments']

    # Plot dendrogram
    fig, ax = plt.subplots(figsize=(16, 5))
    dendrogram(
        Z,
        labels=list(matrix.index),
        leaf_rotation=90,
        leaf_font_size=7,
        color_threshold=0.7 * max(Z[:, 2]),
        ax=ax,
    )
    ax.set_title('Contributor Dendrogram (Ward linkage, Jensen-Shannon distance)')
    ax.set_ylabel('Distance')
    ax.set_xlabel('Contributor')
    plt.tight_layout()
    plt.show()

    print('\nCluster assignments at K=5:')
    if 5 in hier_assignments:
        print(hier_assignments[5].groupby('cluster_id').size().rename('count').reset_index())
else:
    print(f'⚠ Need ≥2 contributors for hierarchical clustering, have {matrix.shape[0]} — skipping.')
    hier_assignments = {1: pd.DataFrame({'contributor': matrix.index, 'cluster_id': 1})}

### 3b. NMF Clustering

In [ ]:
if matrix.shape[0] >= 3 and matrix.shape[1] >= 3:
    nmf_result = cluster_contributors_nmf(matrix, k_range=range(3, 16))

    ks     = [r['k'] for r in nmf_result['results']]
    errors = [r['reconstruction_error'] for r in nmf_result['results']]
    sils   = [r['silhouette_score'] for r in nmf_result['results']]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    ax1.plot(ks, errors, 'o-', color='steelblue')
    ax1.set_xlabel('K (number of groups)')
    ax1.set_ylabel('Reconstruction Error')
    ax1.set_title('NMF: Reconstruction Error vs K')
    ax1.axvline(nmf_result['best_k'], color='crimson', linestyle='--', label=f'Best K={nmf_result["best_k"]}')
    ax1.legend()

    ax2.plot(ks, sils, 'o-', color='darkorange')
    ax2.set_xlabel('K (number of groups)')
    ax2.set_ylabel('Silhouette Score')
    ax2.set_title('NMF: Silhouette Score vs K')
    ax2.axvline(nmf_result['best_k'], color='crimson', linestyle='--', label=f'Best K={nmf_result["best_k"]}')
    ax2.legend()

    plt.tight_layout()
    plt.show()

    print(f'Best K (highest silhouette): {nmf_result["best_k"]}')

    # Extract NMF assignments at best_k
    best_nmf = next(r for r in nmf_result['results'] if r['k'] == nmf_result['best_k'])
    nmf_assignments = pd.DataFrame({
        'contributor': matrix.index,
        'cluster_id': best_nmf['labels'] + 1,
    })
else:
    print(f'⚠ Need ≥3 contributors and ≥3 targets for NMF, have {matrix.shape} — skipping.')
    nmf_result = None
    nmf_assignments = pd.DataFrame({'contributor': matrix.index, 'cluster_id': 1})

### 3c. Bipartite Louvain

In [ ]:
if matrix.shape[0] >= 2:
    # Build bipartite graph: contributor nodes and target nodes
    B = nx.Graph()

    # Add contributor nodes
    B.add_nodes_from(matrix.index.tolist(), bipartite=0)
    # Add target nodes
    B.add_nodes_from(matrix.columns.tolist(), bipartite=1)

    # Add edges weighted by commit count
    for contributor in matrix.index:
        for target in matrix.columns:
            w = matrix.loc[contributor, target]
            if w > 0:
                B.add_edge(contributor, target, weight=int(w))

    print(f'Bipartite graph: {B.number_of_nodes()} nodes, {B.number_of_edges()} edges')

    # Project to contributor-contributor graph
    contributor_nodes = {n for n, d in B.nodes(data=True) if d.get('bipartite') == 0}
    C = nx.bipartite.weighted_projected_graph(B, contributor_nodes)
    print(f'Contributor projection: {C.number_of_nodes()} nodes, {C.number_of_edges()} edges')

    if C.number_of_edges() > 0:
        # Run Louvain
        louvain_partition = community_louvain.best_partition(C, weight='weight', random_state=42)
        louvain_assignments = pd.DataFrame([
            {'contributor': c, 'cluster_id': cluster_id + 1}
            for c, cluster_id in louvain_partition.items()
        ])
        print(f'Louvain found {louvain_assignments["cluster_id"].nunique()} communities')
        print(louvain_assignments.groupby('cluster_id').size().rename('count').reset_index())
    else:
        print('⚠ No edges in contributor projection — skipping Louvain.')
        louvain_assignments = pd.DataFrame({'contributor': matrix.index, 'cluster_id': 1})
else:
    print(f'⚠ Need ≥2 contributors for Louvain, have {matrix.shape[0]} — skipping.')
    louvain_assignments = pd.DataFrame({'contributor': matrix.index, 'cluster_id': 1})

## 4. Consensus — Adjusted Rand Index

In [ ]:
if matrix.shape[0] >= 3 and nmf_result is not None:
    # Pick hierarchical K that best matches NMF best_k
    hier_k = min(hier_assignments.keys(), key=lambda k: abs(k - nmf_result['best_k']))
    hier_df = hier_assignments[hier_k].set_index('contributor')['cluster_id']

    # Align all three on common contributors
    common = (
        set(hier_df.index) &
        set(nmf_assignments['contributor']) &
        set(louvain_assignments['contributor'])
    )
    common = sorted(common)

    hier_labels    = [hier_df.loc[c] for c in common]
    nmf_labels     = [nmf_assignments.set_index('contributor').loc[c, 'cluster_id'] for c in common]
    louvain_labels = [louvain_assignments.set_index('contributor').loc[c, 'cluster_id'] for c in common]

    ari_hier_nmf     = adjusted_rand_score(hier_labels, nmf_labels)
    ari_hier_louvain = adjusted_rand_score(hier_labels, louvain_labels)
    ari_nmf_louvain  = adjusted_rand_score(nmf_labels, louvain_labels)

    ari_matrix = pd.DataFrame(
        [[1.0, ari_hier_nmf, ari_hier_louvain],
         [ari_hier_nmf, 1.0, ari_nmf_louvain],
         [ari_hier_louvain, ari_nmf_louvain, 1.0]],
        index=['Hierarchical', 'NMF', 'Louvain'],
        columns=['Hierarchical', 'NMF', 'Louvain'],
    )

    print('Adjusted Rand Index matrix:')
    print(ari_matrix.round(3))

    fig, ax = plt.subplots(figsize=(6, 4))
    sns.heatmap(ari_matrix, annot=True, fmt='.3f', cmap='Blues', vmin=0, vmax=1,
                linewidths=1, ax=ax)
    ax.set_title('Adjusted Rand Index Between Clustering Methods')
    plt.tight_layout()
    plt.show()
else:
    print('⚠ Not enough contributors for consensus comparison — skipping ARI.')
    # Fall back: use the single available assignment set
    hier_k = min(hier_assignments.keys())

## 5. Labelling

In [ ]:
# Use hierarchical clustering assignments as primary (adjust as preferred)
CHOSEN_K = hier_k
groups_df = hier_assignments[CHOSEN_K].copy()
groups_df = groups_df.rename(columns={'cluster_id': 'group_id'})

# For each group, show top 10 targets by commit share
for gid in sorted(groups_df['group_id'].unique()):
    members = groups_df[groups_df['group_id'] == gid]['contributor'].tolist()
    target_totals = commits_df[commits_df['contributor'].isin(members)].groupby('cmake_target')['commit_count'].sum()
    top10 = target_totals.nlargest(10)
    print(f'\nGroup {gid} ({len(members)} contributors):')
    for t, c in top10.items():
        print(f'  {t:<50} {c:>5} commits')

# Placeholder label mapping — fill in domain names here
group_labels: dict[int, str] = {gid: f'team_{gid}' for gid in sorted(groups_df['group_id'].unique())}
# e.g. group_labels = {1: 'platform', 2: 'rendering', 3: 'networking', ...}
groups_df['group_label'] = groups_df['group_id'].map(group_labels)
print('\nGroup label mapping (edit above to assign domain names):')
print(groups_df.groupby(['group_id', 'group_label']).size().rename('n_contributors').reset_index())

## 6. Ownership Scoring

In [ ]:
ownership_df = compute_ownership(
    commits_df,
    groups_df,
    half_life_days=90,
)
print(f'Ownership rows: {len(ownership_df)}')
print(f'Targets covered: {ownership_df["cmake_target"].nunique()}')
ownership_df.head(10)

In [ ]:
# Pivot ownership to wide format: rows = targets, cols = groups
ownership_wide = ownership_df.pivot_table(
    index='cmake_target', columns='group_id',
    values='ownership_normalised', fill_value=0,
)
ownership_wide.columns = [f'group_{c}' for c in ownership_wide.columns]

# Add owning_group: group with highest normalised ownership
ownership_wide['owning_group_id'] = ownership_df.groupby('cmake_target').apply(
    lambda df: df.loc[df['ownership_normalised'].idxmax(), 'group_id']
)

# Plot ownership heatmap for top 40 targets by compile time
top_targets_by_time = target_df.nlargest(40, 'compile_time_sum_ms')['cmake_target'].tolist()
sub = ownership_wide.reindex(top_targets_by_time).dropna(how='all')
value_cols = [c for c in sub.columns if c.startswith('group_')]

if len(sub) and value_cols:
    fig, ax = plt.subplots(figsize=(12, 8))
    sns.heatmap(sub[value_cols], ax=ax, cmap='Blues', vmin=0, vmax=1,
                yticklabels=True, linewidths=0.3,
                cbar_kws={'label': 'Normalised ownership'})
    ax.set_title('Target Ownership (top 40 targets by compile time)')
    plt.tight_layout()
    plt.show()

## 7. Ownership Concentration (HHI)

In [ ]:
# Herfindahl-Hirschman Index per target: sum of squared normalised ownership shares
# HHI == 1.0 → sole owner;  HHI → 0 → perfectly contested
hhi_per_target = ownership_df.groupby('cmake_target').apply(
    lambda df: (df['ownership_normalised'] ** 2).sum()
).rename('hhi')

contested_threshold = 0.5
contested = hhi_per_target[hhi_per_target < contested_threshold]

print(f'Targets with HHI < {contested_threshold} (contested): {len(contested)}')
print(contested.sort_values().head(20).to_frame())

fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(hhi_per_target, bins=40, color='steelblue', edgecolor='white')
ax.axvline(contested_threshold, color='crimson', linestyle='--', label=f'Contested threshold ({contested_threshold})')
ax.set_xlabel('HHI')
ax.set_ylabel('Number of targets')
ax.set_title('Ownership Concentration (HHI) per Target')
ax.legend()
plt.tight_layout()
plt.show()

## 8. Bus Factor

In [ ]:
# Load git history detail for bus factor computation
git_detail_path = cfg.raw_data_dir / 'git_history_detail.csv'
git_detail = pd.read_csv(git_detail_path)

# Map source_file -> cmake_target via file_df
file_to_target = dict(zip(file_df['source_file'], file_df['cmake_target']))
git_detail['cmake_target'] = git_detail['source_file'].map(file_to_target)
git_detail = git_detail.dropna(subset=['cmake_target'])

bus_df = compute_bus_factor(git_detail, groups_df, recent_months=3)
print(f'Bus factor rows: {len(bus_df)}')

low_bus = bus_df[bus_df['bus_factor'] <= 1].copy()
print(f'Targets with bus_factor <= 1 (at risk): {low_bus["cmake_target"].nunique()}')

# Merge with compile time for risk prioritisation
low_bus = low_bus.merge(target_df[['cmake_target', 'compile_time_sum_ms']], on='cmake_target', how='left')
print('\nHighest-risk targets (low bus factor, high compile time):')
print(
    low_bus.sort_values('compile_time_sum_ms', ascending=False)
    [['cmake_target', 'group_id', 'bus_factor', 'compile_time_sum_ms']]
    .head(20)
    .to_string(index=False)
)

In [ ]:
# Distribution of bus factors
fig, ax = plt.subplots(figsize=(8, 4))
bus_max = bus_df.groupby('cmake_target')['bus_factor'].max()
bus_max.value_counts().sort_index().plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.axvline(0.5, color='crimson', linestyle='--', label='bus_factor=1 threshold')
ax.set_xlabel('Bus Factor (max across groups)')
ax.set_ylabel('Number of Targets')
ax.set_title('Bus Factor Distribution per Target')
plt.tight_layout()
plt.show()

## 9. Cross-Team Dependency Map

In [ ]:
# Map each target to its owning group
target_to_group = ownership_wide['owning_group_id'].dropna().to_dict()
n_groups = len(set(target_to_group.values()))

if n_groups >= 2:
    # Classify each edge as within-team or cross-team
    edge_df_groups = edge_df.copy()
    edge_df_groups['src_group'] = edge_df_groups['source_target'].map(target_to_group)
    edge_df_groups['dst_group'] = edge_df_groups['dest_target'].map(target_to_group)
    edge_df_groups = edge_df_groups.dropna(subset=['src_group', 'dst_group'])

    # Direct edges only
    direct_edges = edge_df_groups[edge_df_groups['is_direct'].eq(True)]
    cross_team   = direct_edges[direct_edges['src_group'] != direct_edges['dst_group']]

    total_direct = len(direct_edges)
    total_cross  = len(cross_team)
    print(f'Total direct edges (mapped): {total_direct}')
    if total_direct > 0:
        print(f'Cross-team edges            : {total_cross} ({100*total_cross/total_direct:.1f}%)')

    # Team-level edge summary: cross-team edge counts between pairs
    team_summary = (
        cross_team.groupby(['src_group', 'dst_group'])
        .size()
        .rename('cross_edges')
        .reset_index()
        .sort_values('cross_edges', ascending=False)
    )
    print('\nCross-team edge pairs (top 20):')
    print(team_summary.head(20).to_string(index=False))
else:
    print(f'Only {n_groups} group(s) — cross-team analysis requires ≥2 groups, skipping.')
    team_summary = pd.DataFrame(columns=['src_group', 'dst_group', 'cross_edges'])

In [ ]:
if len(team_summary) > 0:
    # Visualise as a team-level dependency matrix
    all_groups = sorted(set(team_summary['src_group']) | set(team_summary['dst_group']))
    team_matrix = pd.DataFrame(0, index=all_groups, columns=all_groups)
    for _, row in team_summary.iterrows():
        team_matrix.loc[row['src_group'], row['dst_group']] = row['cross_edges']

    fig, ax = plt.subplots(figsize=(8, 7))
    sns.heatmap(team_matrix, annot=True, fmt='d', cmap='Oranges', linewidths=0.5, ax=ax)
    ax.set_title('Cross-Team Dependency Edges (source team → dest team)')
    ax.set_xlabel('Destination Team')
    ax.set_ylabel('Source Team')
    plt.tight_layout()
    plt.show()
else:
    print('No cross-team edges to visualise.')

## 10. Save Outputs

In [ ]:
import pyarrow as pa
import pyarrow.parquet as pq

# contributor_groups.parquet: (contributor, group_id, group_label)
groups_out = groups_df[['contributor', 'group_id', 'group_label']].copy()
groups_path = cfg.processed_data_dir / 'contributor_groups.parquet'
pq.write_table(
    pa.Table.from_pandas(groups_out, preserve_index=False),
    groups_path,
    compression='snappy',
)
print(f'Wrote {len(groups_out)} rows -> {groups_path}')

# target_ownership.parquet: (cmake_target, group_id, ownership_score, ownership_normalised, owning_group_id, hhi)
ownership_out = ownership_df.copy()
ownership_out = ownership_out.merge(
    ownership_wide[['owning_group_id']].reset_index(),
    on='cmake_target', how='left',
)
ownership_out = ownership_out.merge(
    hhi_per_target.reset_index(),
    on='cmake_target', how='left',
)

ownership_path = cfg.processed_data_dir / 'target_ownership.parquet'
pq.write_table(
    pa.Table.from_pandas(ownership_out, preserve_index=False),
    ownership_path,
    compression='snappy',
)
print(f'Wrote {len(ownership_out)} rows -> {ownership_path}')
print('\nDone.')